# Laboratorio 4 · EDA de INVACOM_DW
**Universidad Técnica Nacional · ISW-911 Minería de Datos**

**Integrantes:** Gaudy Yuliana Chacón Alpizar, Joab Mauricio Montero Montero, Ignacio Araya Rojas y Stuarth Fabricio Miranda Rojas.

Fuente: esquema estrella INVACOM_DW del Proyecto 1. Se conserva una línea por pedido y producto. La selección del DW sustituye el requisito de datos abiertos según el alcance solicitado por el grupo.

## 1-3. Fuente Oracle y lectura desde Git
La consulta de solo lectura en sql/01_extraccion_invacom_dw.sql integra FACT_VENTAS y las tres dimensiones. El exportador genera el CSV y su manifiesto con SHA-256 y conciliación. El notebook lee ambos desde GitHub; no recurre a Northwind si faltan. Publicar primero ambos archivos. Se puede fijar REF_GIT al commit de la extracción para reproducibilidad.

In [ ]:
# Integrantes: Gaudy Yuliana Chacón Alpizar, Joab Mauricio Montero Montero,
# Ignacio Araya Rojas y Stuarth Fabricio Miranda Rojas.
import hashlib
import io
import re
import unicodedata
import urllib.request
from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor

%matplotlib inline
pd.set_option("display.max_columns", 22)
pd.set_option("display.max_rows", 25)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 10})
AZUL, NARANJA = "#25798c", "#c77735"

In [ ]:
REF_GIT = "main"
URL_BASE_GIT = f"https://raw.githubusercontent.com/nachoar24/workshop-4-mineria-datos/{REF_GIT}/datos/invacom_dw/"
import json

def descargar(nombre):
    with urllib.request.urlopen(URL_BASE_GIT + nombre, timeout=60) as respuesta:
        return respuesta.read()

manifest = json.loads(descargar("manifest.json"))
contenido = descargar("invacom_dw_ventas.csv")
assert manifest["source"] == "Oracle INVACOM_DW"
assert hashlib.sha256(contenido).hexdigest() == manifest["sha256"]
datos_origen = pd.read_csv(io.BytesIO(contenido), dtype="string", keep_default_na=False, na_values=[""])
assert len(datos_origen) == manifest["rows"]
print("Extracción Oracle:", manifest["extracted_at_utc"], datos_origen.shape)

In [ ]:
assert not datos_origen.duplicated(["ID_PEDIDO", "ID_PRODUCTO"]).any()
assert datos_origen["ID_FACT_VENTA"].is_unique
print("Pedidos únicos:", datos_origen["ID_PEDIDO"].nunique())

## 4. Características principales

Se inspeccionan dimensiones, ejemplos, tipos, valores distintos y estadísticas básicas.
Los CSV se leyeron como texto para conservar los códigos y mostrar explícitamente
sus conversiones posteriores. Los estadísticos numéricos se presentan después de
convertir los tipos.

In [ ]:
print("Filas y columnas", datos_origen.shape)
display(datos_origen.head())
display(datos_origen.tail(3))
display(datos_origen.sample(3, random_state=42))
datos_origen.info()
display(datos_origen.describe(include="all").T)
display(datos_origen.nunique().rename("valores_distintos").to_frame())
print("Claves de línea duplicadas", datos_origen.duplicated(["orderID", "productID"]).sum())

## 5. Renombrado de columnas a español
El DW ya emplea nombres en español. Se normalizan a minúsculas y se uniforma ANIO como ano.

In [ ]:
nombres_espanol = {c: c.lower() for c in datos_origen.columns}
nombres_espanol["ANIO"] = "ano"
display(pd.DataFrame(nombres_espanol.items(), columns=["nombre_original","nombre_espanol"]))
datos = datos_origen.rename(columns=nombres_espanol).copy()
datos = datos.apply(lambda c: c.str.strip()).replace(r"^\s*$", pd.NA, regex=True)
assert datos.columns.is_unique

## 6. Nulos con evidencia gráfica
El DW ya fue depurado mediante ETL; no se introducen nulos artificiales. Se cuantifican las ausencias reales, incluso cuando son cero.

In [ ]:
nulos_antes = datos.isna().sum()
informe_nulos = pd.DataFrame({"cantidad": nulos_antes,
                             "porcentaje": nulos_antes / len(datos) * 100})
display(informe_nulos)
fig, eje = plt.subplots(figsize=(10, 5))
nulos_antes.sort_values().plot.barh(ax=eje, color=NARANJA)
eje.set(title="Nulos por variable antes del tratamiento", xlabel="Celdas nulas", ylabel="")
fig.tight_layout()
plt.show()
print("Total de celdas nulas", int(nulos_antes.sum()))

## 9. Conversión de tipos
Claves y cantidades enteras, importes decimales y fecha ISO. Los importes almacenados son la referencia. Precio y tasa derivados se usan exclusivamente para descripción, no como predictores, para evitar fuga de información.

In [ ]:
for c in ["id_fact_venta","id_pedido","id_producto","cantidad_vendida","ano","mes"]:
    datos[c] = pd.to_numeric(datos[c], errors="raise").astype("int64")
columnas_importes = ["importe_bruto","importe_descuento","importe_neto"]
for c in columnas_importes:
    datos[c] = pd.to_numeric(datos[c], errors="raise")
datos["fecha_pedido"] = pd.to_datetime(datos["fecha_pedido"], format="%Y-%m-%d", errors="raise")
assert not datos[["fecha_pedido","cantidad_vendida"] + columnas_importes].isna().any().any()
assert datos["cantidad_vendida"].gt(0).all()
assert datos["importe_bruto"].ge(0).all()
assert datos["importe_descuento"].between(0, datos["importe_bruto"]).all()
assert datos["importe_neto"].ge(0).all()
assert datos["ano"].eq(datos["fecha_pedido"].dt.year).all()
assert datos["mes"].eq(datos["fecha_pedido"].dt.month).all()
datos["dia_semana"] = datos["fecha_pedido"].dt.dayofweek
datos["precio_unitario"] = datos["importe_bruto"] / datos["cantidad_vendida"]
datos["tasa_descuento"] = datos["importe_descuento"] / datos["importe_bruto"].replace(0, np.nan)
display(datos.dtypes.to_frame("tipo"))
display(datos.select_dtypes("number").describe())

## 7. Valores atípicos con evidencia gráfica

Se utiliza IQR para señalar observaciones fuera de Q1 − 1,5 × IQR y Q3 + 1,5 × IQR.
Un pedido de muchas unidades o un producto caro puede ser comercialmente válido.
Se conservan estos valores y se informa su cantidad. La cantidad vendida será la
variable dependiente, por lo que reemplazar sus valores altos alteraría el fenómeno
de demanda que se pretende estudiar.

In [ ]:
def detectar_atipicos_iqr(serie):
    '''Identifica observaciones extremas de una medida comercial sin modificarlas.'''
    q1, q3 = serie.dropna().quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((serie < inferior) | (serie > superior)).fillna(False), inferior, superior

medidas = ["cantidad_vendida", "precio_unitario", "tasa_descuento"] + columnas_importes
informe_atipicos = []
fig, ejes = plt.subplots(2, 3, figsize=(13, 8))
for eje, columna in zip(ejes.flat, medidas):
    marca, inferior, superior = detectar_atipicos_iqr(datos[columna])
    informe_atipicos.append({"variable": columna, "inferior": inferior, "superior": superior,
                             "atipicos": int(marca.sum())})
    eje.boxplot(datos[columna].dropna(), orientation="horizontal",
                flierprops={"markersize": 3, "markerfacecolor": NARANJA})
    eje.set(title=columna, xlabel="Valor observado", yticks=[])
fig.suptitle("Distribución y valores señalados por IQR")
fig.tight_layout()
plt.show()
display(pd.DataFrame(informe_atipicos))

## 8. Imputación
Las categorías faltantes se representan como Sin información. Las métricas obligatorias se validan, no se inventan. Una tasa indefinida por bruto cero se representa como cero únicamente para descripción, y se registra el cambio. Se conserva la función de mediana como alternativa para variables numéricas opcionales. Los atípicos válidos se conservan.

In [ ]:
def imputar_categoricas(tabla, columnas):
    resultado = tabla.copy()
    resultado[columnas] = resultado[columnas].fillna("Sin información")
    return resultado

def imputar_numericas_mediana(tabla, columnas):
    resultado = tabla.copy()
    resultado[columnas] = resultado[columnas].fillna(resultado[columnas].median())
    return resultado

obligatorias = ["id_fact_venta","id_pedido","id_producto","id_cliente","fecha_pedido","cantidad_vendida"] + columnas_importes
assert not datos[obligatorias].isna().any().any()
categoricas = ["producto","cliente","ciudad_cliente","pais_cliente","categoria"]
datos_limpios = imputar_categoricas(datos, categoricas)
print("Tasas indefinidas:", datos_limpios["tasa_descuento"].isna().sum())
datos_limpios["tasa_descuento"] = datos_limpios["tasa_descuento"].fillna(0)
datos_limpios = imputar_numericas_mediana(datos_limpios, ["precio_unitario"])
comparacion = pd.DataFrame({"antes": nulos_antes, "despues": datos_limpios[nulos_antes.index].isna().sum()})
comparacion.plot.barh(figsize=(10,7), color=[NARANJA,AZUL])
plt.title("Nulos antes y después"); plt.tight_layout(); plt.show()
assert not datos_limpios.isna().any().any()

## Conciliación con Oracle
Se utilizan los importes almacenados en FACT_VENTAS. Se comparan los totales con el manifiesto de extracción; las referencias históricas del TKT-17 son informativas.

In [ ]:
assert np.allclose(datos_limpios["importe_bruto"]-datos_limpios["importe_descuento"],datos_limpios["importe_neto"],atol=0.005,rtol=0)
totales = {"lineas":len(datos_limpios),"unidades":datos_limpios["cantidad_vendida"].sum(),"bruto":datos_limpios["importe_bruto"].sum(),"descuento":datos_limpios["importe_descuento"].sum(),"neto":datos_limpios["importe_neto"].sum()}
for k,v in totales.items():
    assert np.isclose(float(v),float(manifest["totals"][k]),atol=0.005,rtol=0), k
display(pd.Series(totales,name="total conciliado"))
fig,ejes = plt.subplots(1,2,figsize=(12,4))
datos_limpios.groupby(datos_limpios["fecha_pedido"].dt.to_period("M"))["importe_neto"].sum().plot(ax=ejes[0],marker="o")
ejes[0].set_title("Ventas netas por mes")
datos_limpios.groupby("categoria")["cantidad_vendida"].sum().plot.barh(ax=ejes[1])
ejes[1].set_title("Unidades por categoría")
fig.tight_layout(); plt.show()

## 10. Codificación categórica
One-hot para producto, categoría, país y ciudad. Mes y día de semana se codifican cíclicamente. Se excluyen identificadores e importes y sus derivados de los predictores.

In [ ]:
categoricas_modelo = ["producto", "categoria", "pais_cliente", "ciudad_cliente"]
indicadores = pd.get_dummies(datos_limpios[categoricas_modelo], prefix_sep="__", dtype="uint8")

def normalizar_nombre(nombre):
    '''Conserva nombres de columnas en español con caracteres aptos para el CSV.'''
    texto = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9_]+", "_", texto.lower()).strip("_")

indicadores.columns = [normalizar_nombre(nombre) for nombre in indicadores.columns]
predictores = datos_limpios[["ano"]].copy()
for columna, periodo, inicio in [("mes", 12, 1), ("dia_semana", 7, 0)]:
    angulo = 2 * np.pi * (datos_limpios[columna] - inicio) / periodo
    predictores[columna + "_seno"] = np.sin(angulo)
    predictores[columna + "_coseno"] = np.cos(angulo)
predictores = pd.concat([predictores, indicadores], axis=1)
objetivo = datos_limpios["cantidad_vendida"].copy()
assert predictores.columns.is_unique
print("Matriz de predictores", predictores.shape)
display(predictores.head())

## 11. Estandarización
Se estandariza año mediante StandardScaler; los indicadores binarios y cíclicos conservan su escala. Cantidad vendida conserva su unidad. En una evaluación posterior se ajustará el preprocesamiento solo con entrenamiento.

In [ ]:
columnas_estandarizar = ["ano"]
escalador = StandardScaler()
valores_z = escalador.fit_transform(predictores[columnas_estandarizar])
predictores_z = predictores.drop(columns=columnas_estandarizar).copy()
for posicion, columna in enumerate(columnas_estandarizar):
    predictores_z[columna + "_z"] = valores_z[:, posicion]
parametros = pd.DataFrame({"media_original": escalador.mean_, "escala_original": escalador.scale_,
                           "media_z": valores_z.mean(axis=0), "desviacion_z": valores_z.std(axis=0)},
                          index=columnas_estandarizar)
display(parametros.round(6))
fig, ejes = plt.subplots(1, 2, figsize=(11, 3))
ejes[0].hist(predictores["ano"], bins=8, color=AZUL)
ejes[0].set(title="Año", xlabel="Precio", ylabel="Líneas")
ejes[1].hist(predictores_z["ano_z"], bins=35, color=AZUL)
ejes[1].set(title="Año estandarizado", xlabel="Desviaciones estándar", ylabel="Líneas")
fig.tight_layout()
plt.show()

## 12. Correlación y evidencia gráfica

Se calcula Pearson sobre las medidas comerciales. Los identificadores se excluyen
de esta interpretación numérica. Las asociaciones entre cantidad, bruto y neto
también reflejan las fórmulas que los relacionan. Por eso los importes calculados
a partir de cantidad no se incluyen entre sus predictores.

In [ ]:
columnas_correlacion = ["cantidad_vendida", "precio_unitario", "tasa_descuento",
                        "importe_bruto", "importe_descuento", "importe_neto"]
matriz = datos_limpios[columnas_correlacion].corr()
fig, eje = plt.subplots(figsize=(8, 6))
imagen = eje.imshow(matriz, cmap="RdBu_r", vmin=-1, vmax=1)
eje.set_xticks(range(len(matriz)), matriz.columns, rotation=40, ha="right")
eje.set_yticks(range(len(matriz)), matriz.columns)
for fila in range(len(matriz)):
    for columna in range(len(matriz)):
        valor = matriz.iloc[fila, columna]
        eje.text(columna, fila, f"{valor:.2f}", ha="center", va="center",
                 color="white" if abs(valor) > 0.55 else "#18232b")
eje.set_title("Correlación de las medidas comerciales de INVACOM")
fig.colorbar(imagen, ax=eje, label="r de Pearson")
fig.tight_layout()
plt.show()
display(matriz["cantidad_vendida"].drop("cantidad_vendida").rename("correlacion").to_frame().round(4))
correlaciones = predictores_z.corrwith(objetivo).dropna()
principales = correlaciones.reindex(correlaciones.abs().sort_values(ascending=False).head(10).index)
principales.sort_values().plot.barh(figsize=(10, 5), color=AZUL)
plt.axvline(0, color="#596872", linewidth=1)
plt.title("Diez asociaciones de predictores con cantidad vendida")
plt.xlabel("Correlación de Pearson")
plt.tight_layout()
plt.show()
display(principales.rename("correlacion").to_frame().round(4))

## 13. Variable dependiente y modelo candidato
Cantidad vendida por línea de pedido. Se propone DecisionTreeRegressor para relaciones no lineales entre producto, categoría, ubicación y calendario. Importes y derivados quedan excluidos por riesgo de fuga. La predicción supone conocidos los atributos de la línea antes de conocer la cantidad; no equivale a pronosticar ventas totales. Para evaluar posteriormente, separar por pedido o fecha, ajustar transformaciones solo con entrenamiento y medir MAE/RMSE. El laboratorio solicita seleccionar y justificar, no entrenar.

In [ ]:
modelo_candidato = DecisionTreeRegressor(max_depth=5, min_samples_leaf=15, random_state=42)
print("Variable dependiente", objetivo.name)
display(objetivo.describe().to_frame())
print(modelo_candidato)
assert "cantidad_vendida" not in predictores_z
assert not set(columnas_importes).intersection(predictores_z.columns)

## 14. Guardado del conjunto procesado
Se exportan predictores, objetivo e identificadores e importes numéricos para auditoría. No usar columnas de auditoría para entrenar.

In [ ]:
auditoria = datos_limpios[["id_fact_venta", "id_pedido", "id_producto"] + columnas_importes].copy()
for columna in ["id_pedido", "id_producto"]:
    auditoria[columna] = pd.to_numeric(auditoria[columna], errors="raise").astype("int32")
datos_procesados = pd.concat([auditoria, predictores_z, objetivo], axis=1)
assert len(datos_procesados) == len(datos_origen)
assert not datos_procesados.duplicated(["id_pedido", "id_producto"]).any()
assert not datos_procesados.isna().any().any()
assert np.isfinite(datos_procesados.to_numpy(dtype=float)).all()
datos_procesados.to_csv("data_process.csv", index=False, encoding="utf-8", float_format="%.10g")
comprobacion = pd.read_csv("data_process.csv")
assert comprobacion.shape == datos_procesados.shape
assert comprobacion.columns.tolist() == datos_procesados.columns.tolist()
assert np.allclose(comprobacion, datos_procesados, rtol=1e-8, atol=1e-9)
print("Archivo guardado data_process.csv")
print("Dimensiones", datos_procesados.shape)
print("Unidades conciliadas", datos_procesados["cantidad_vendida"].sum())
print("Venta neta conciliada", round(datos_procesados["importe_neto"].sum(), 2))
print("Nulos finales", int(datos_procesados.isna().sum().sum()))
display(datos_procesados.head())

## 15-16. Entrega
Entregar lab4_IC2025.ipynb con todas las celdas ejecutadas y data_process.csv regenerado. El manifiesto y CSV de Oracle deben estar publicados previamente.